# MDLM Pipeline Walkthrough (Repo-Aware)

This notebook mirrors the repository implementation of **Simple and Effective Masked Diffusion Language Models**. Each section points to the relevant source files so readers can connect the paper's method to `main.py`, `dataloader.py`, `diffusion.py`, `noise_schedule.py`, and associated utilities.


## 1. Entry Point & Configuration

`main.py` wires Hydra to dispatch between training, perplexity evaluation, or sampling via `mode` (`_train`, `_ppl_eval`, `generate_samples`). The configuration tree is rooted at `configs/config.yaml` and expands defaults for the dataset, backbone, diffusion noise, optimization, and sampling strategy. Hydra resolvers (declared in `main.py`) also provide conveniences such as path interpolation.


In [ ]:
import omegaconf
from omegaconf import OmegaConf

# Load the same default tree Hydra would inject into main.py
base_cfg = OmegaConf.load("configs/config.yaml")
print("Top-level keys:", list(base_cfg.keys()))
print("Backbone type:", base_cfg.model.backbone)
print("Noise schedule type:", base_cfg.noise.type)


## 2. Tokenization, Masking, and Batching

`dataloader.py` constructs a tokenizer (Text8 by default, but HuggingFace tokenizers are also supported), streams the dataset, and prepares `(input_tokens, output_tokens, attention_mask)` pairs. A mask token is injected when the tokenizer lacks one.

`_maybe_sub_sample` in `diffusion.py` optionally crops or shifts sequences:
* **SUBS / D3PM**: keeps the full sequence and (if `subs_masking`) enforces BOS/EOS on the boundaries to match evaluation.
* **Autoregressive baseline**: shifts tokens by one position to predict the next token.

The returned `attention_mask` gates losses per position, matching how `Loss.loss` and `Loss.nlls` are computed in the trainer.


In [ ]:
from dataloader import Text8Tokenizer

# Light-weight tokenizer demo mirroring the repository's default setup
text = "masked diffusion language models"
tokenizer = Text8Tokenizer()
encoded = tokenizer(text).input_ids
mask_index = tokenizer.mask_token_id or len(tokenizer)
print("Encoded ids:", encoded)
print("Decoded text:", tokenizer.decode(encoded))
print("Mask token id used during diffusion:", mask_index)


## 3. Noise Schedule (Continuous Time)

`noise_schedule.py` supplies multiple schedules. The default **log-linear** schedule (used in `config.yaml`) is
\[\sigma(t) = -\log\big(1 - (1-\varepsilon) t\big), \quad \dot\sigma(t) = \frac{1-\varepsilon}{1 - (1-\varepsilon) t}\]
with `eps = 1e-3` to avoid degeneracy at \(t=0\). The sampler and loss functions consume either `(sigma, dsigma)` or a reparameterized time \(t\) depending on `change_of_variables` and `importance_sampling` flags.


In [ ]:
import torch
from noise_schedule import LogLinearNoise

noise = LogLinearNoise(eps=1e-3)
steps = torch.linspace(0, 1, 5)
sigma = noise.total_noise(steps)
dsigma = noise.rate_noise(steps)
for t, s, ds in zip(steps.tolist(), sigma.tolist(), dsigma.tolist()):
    print(f"t={t:.2f} -> sigma={s:.4f}, dsigma/dt={ds:.4f}")


## 4. Forward Diffusion: \(q(x_t \mid x_0)\)

* `_sample_t` draws diffusion times uniformly; when `antithetic_sampling=True` the batch is paired with symmetric times for variance reduction.
* For discrete schedules (`T>0`), times are quantized to `1/T, 2/T, …, 1` before diffusion.
* With continuous time SUBS, the masking probability is \(\text{move\_chance} = 1 - e^{-\sigma(t)}\). With `change_of_variables=True`, the schedule is reparameterized to \(t \mapsto f(t) = \log(1-e^{-\sigma_{\min}}) + t\,(\log(1-e^{-\sigma_{\max}})-\log(1-e^{-\sigma_{\min}}))\) before computing \(\text{move\_chance}=e^{f(t)}\).

`Diffusion.q_xt` masks tokens independently with that probability:
\[x_t = \mathrm{mask\_index}\;\mathbb{1}[u < \text{move\_chance}] + x_0\;\mathbb{1}[u \geq \text{move\_chance}]\]


In [ ]:
import torch

def q_xt(x, move_chance, mask_index):
    move_indices = torch.rand_like(x, dtype=torch.float32) < move_chance
    return torch.where(move_indices, mask_index, x)

x0 = torch.tensor([encoded[:8]])
move_chance = 1 - torch.exp(-noise.total_noise(torch.tensor([0.5])))
xt = q_xt(x0, move_chance[:, None], mask_index)
print("sigma(0.5)=", noise.total_noise(torch.tensor(0.5)).item())
print("move chance:", move_chance.item())
print("x0 -> xt:", x0.tolist(), xt.tolist())


## 5. Training Objectives

All objectives are applied per token and masked by `attention_mask` before aggregation in `_loss`.

### SUBS (continuous time, `T=0`)
For the standard path, the loss is
\[\mathcal{L}_{\text{SUBS}} = -\log p_\theta(x_0 \mid x_t) \cdot \frac{\dot\sigma(t)}{e^{\sigma(t)}-1}\]
When `change_of_variables` or `importance_sampling` is enabled, the repository collapses the weight to the constant
\[\log(1 - e^{-\sigma_{\min}})\]
and multiplies it with the log-probability (note the sign flip in code because `\log p_\theta` is negative).

### D3PM / SUBS (discrete, `T>0`)
`_d3pm_loss` uses cross-entropy against `x0` for \(t>0\). When `parameterization="d3pm"`, a reconstruction term at \(t=0\) is added to encourage clean-token fidelity; for `parameterization="subs"`, this term is omitted.

### SEDD
`_score_entropy` computes the entropy functional on masked positions with weight \(\dot\sigma\):
\[\mathcal{L}_{\text{SEDD}} = \dot\sigma(t) \Big( \sum_{k\neq \text{mask}} q(k\mid x_t) - q(x_0\mid x_t)\,\log q(x_0\mid x_t) + c(t) \Big)\]
where \(q\) is the model score and \(c(t)=\tfrac{1}{\exp(\sigma)-1}\big(\log\tfrac{1}{\exp(\sigma)-1}-1\big)\) matches `_score_entropy`.


In [ ]:
import torch

# Toy log-probs for the true token at sigma=0.5
true_logp = torch.log(torch.tensor([[0.7, 0.3]]))  # two-token toy vocab
sigma_t = noise.total_noise(torch.tensor([0.5]))
dsigma_t = noise.rate_noise(torch.tensor([0.5]))
weight = dsigma_t / torch.expm1(sigma_t)
subs_loss = -true_logp[:, :1] * weight[:, None]
print("Weight (dsigma / expm1(sigma)):", weight.item())
print("SUBS per-token loss:", subs_loss.squeeze().item())


## 6. Reverse Process & Sampling

Sampling starts from an all-mask prior (`_sample_prior`) and repeatedly denoises.

* **DDPM predictor** (`_ddpm_update`):
  \[q(x_{t-\Delta t}\mid x_t) \propto p_\theta(x_0 \mid x_t) \cdot (\alpha_t - \alpha_{t-\Delta t}),\quad \alpha_t = 1 - e^{-\sigma_t}\]
  The mask token receives mass \(\alpha_{t-\Delta t}\) and `copy_flag` keeps already unmasked tokens frozen.
* **DDPM caching** (`_ddpm_caching_update`): identical transition but reuses cached \(p_\theta(x_0\mid x_t)\) across steps for SUBS guidance when `sampling.predictor="ddpm_cache"`.
* **Analytic predictor**: available for continuous-time schedules via `_analytic_update`/`_denoiser_update`.

Autoregressive sampling uses `_ar_sampler` when `parameterization="ar"`.


In [ ]:
# Minimal reproduction of the categorical transition used in _ddpm_update
import torch.nn.functional as F

sigma_t = noise.total_noise(torch.tensor([1.0]))
sigma_s = noise.total_noise(torch.tensor([0.8]))
move_t = 1 - torch.exp(-sigma_t)
move_s = 1 - torch.exp(-sigma_s)

# Fake model prediction: prefer token 0 over 1
log_p_x0 = torch.log_softmax(torch.tensor([[[2.0, 0.5, -1.0]]]), dim=-1)
q_xs = log_p_x0.exp() * (move_t - move_s)[:, None, None]
q_xs[:, :, mask_index] = move_s[:, None]
probs = q_xs / q_xs.sum(dim=-1, keepdim=True)
print("Transition prob to each token (including mask):", probs[0,0].tolist())


## 7. Evaluation Metrics

Training logs three token-level metrics via `torchmetrics` (see `Loss.nlls` and metric collections in `Diffusion.__init__`):
* **NLL**: mean negative log-likelihood across masked positions.
* **Bits-per-dimension (BPD)**: \(\text{NLL} / \log 2\).
* **Perplexity (PPL)**: \(\exp(\text{NLL})\).

For generated text, `compute_generative_perplexity` retokenizes samples with an external AR model (e.g., GPT-2) and aggregates cross-entropy up to the first EOS token, matching the zero-shot evaluation protocol described in the paper.
